# Lab W1c (Breadth): MLP dari Nol dengan Numpy

## Pre-flight Checklist

> [!IMPORTANT]
> Lab ini **breadth opsional** untuk Breadth Check (keluarga MLP). Tidak wajib selesai sebelum W2, tapi sangat dianjurkan saat Anda ingin paham apa yang sebenarnya terjadi di balik `loss.backward()`.

**Yang Anda butuhkan sebelum mulai:**
- Bab W1 sudah dibaca, terutama §2.1 (MLP sebagai pengubah bentuk tensor) dan §2.3 (cara kerja backprop).
- Terbiasa dengan operasi matriks NumPy: `A @ B`, broadcasting, argumen `axis=`.
- Ingat chain rule dari §0.5.4 Pendahuluan (rantai turunan = perkalian turunan).
- Lampiran A.1 tersedia jika butuh derivasi 7-langkah formal.

**Yang akan Anda hasilkan di akhir lab:**
- MLP 2-layer (784 → 256 → 10) yang ditulis dari nol di NumPy.
- Forward dan backward pass manual; tidak ada `loss.backward()`.
- Verifikasi finite-difference: gradient analitik vs numerik harus selaras dalam toleransi 1e-3.
- Loss curve sebanding dengan PyTorch autograd (parity check).

**Sumber belajar:**
- **Hardware:** CPU cukup. Training 5 epoch MNIST di NumPy ~1-2 menit.
- **Estimasi waktu kerja:** 2-3 jam termasuk derivasi manual dan refleksi.

**Tujuan pedagogis:** mahasiswa menulis forward dan backward pass MLP dua-layer secara manual, lalu melatihnya pada MNIST. Tidak ada PyTorch autograd di sel inti; hanya numpy. Sel terakhir membandingkan dengan `SimpleMLP` PyTorch sebagai parity check.

**Pendamping:** Bab W1 (`01_W1_Tabular_Output_Heads.md`) dan Lampiran A.1 di `14_Lampiran.md` untuk derivasi 7-langkah.

## Fase 1: Memuat dan Memeriksa Data

Sebelum menulis kode model, kita periksa dulu data yang akan dipakai: ukuran, bentuk, dan rentang nilainya.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
print('NumPy versi:', np.__version__)

In [ ]:
# MNIST: 60.000 gambar angka tulisan tangan, tiap gambar 28x28 piksel
from torchvision import datasets, transforms

tfm = transforms.ToTensor()
train_ds = datasets.MNIST(root='./data', train=True,  download=True, transform=tfm)
test_ds  = datasets.MNIST(root='./data', train=False, download=True, transform=tfm)

X_train = train_ds.data.numpy().astype('float32').reshape(-1, 784) / 255.0
y_train = train_ds.targets.numpy().astype('int64')
X_test  = test_ds.data.numpy().astype('float32').reshape(-1, 784)  / 255.0
y_test  = test_ds.targets.numpy().astype('int64')

print('Total data latih  :', len(X_train), 'gambar')
print('Total data uji    :', len(X_test),  'gambar')
print('Shape satu gambar :', X_train[0].shape, '<-- 784 = 28x28 piksel yang diratakan')
print('Rentang nilai piksel: min =', X_train.min(), '| maks =', X_train.max())

In [ ]:
import matplotlib.pyplot as plt

x_satu = X_train[0]
y_satu = y_train[0]

print('Shape satu gambar:', x_satu.shape)
print('Label            :', y_satu)
print('10 nilai piksel pertama:', x_satu[:10].round(3))

plt.imshow(x_satu.reshape(28, 28), cmap='gray')
plt.title('Label: ' + str(y_satu))
plt.axis('off')
plt.show()

## Fase 2: Membangun Komponen Tunggal

Kita bangun tiap bagian satu per satu. Setiap kali mendefinisikan fungsi baru, kita uji dulu dengan input kecil yang hasilnya bisa kita periksa secara manual.

In [ ]:
# ReLU: angka negatif jadi 0, angka positif tetap
angka_contoh = np.array([-3.0, -1.0, 0.0, 2.5, 4.0])
print('Sebelum ReLU:', angka_contoh)
print('Sesudah ReLU:', np.maximum(0, angka_contoh))

In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    # Turunan ReLU: 1 jika input positif, 0 jika tidak
    return (z > 0).astype(z.dtype)

z_kecil = np.array([-2.0, -0.5, 0.3, 1.8])
print('Input  :', z_kecil)
print('ReLU   :', relu(z_kecil))
print('Gradien:', relu_grad(z_kecil), '<-- 1 = ada gradient, 0 = tidak')

### 2.2 Softmax + Cross-Entropy: Mengukur Seberapa Salah Model

Model kita akan mengeluarkan 10 angka mentah - satu per kelas angka (0-9). Angka-angka ini belum bisa disebut probabilitas karena bisa negatif dan jumlahnya tidak harus 1.

Softmax mengubah 10 angka mentah itu menjadi 10 probabilitas yang jumlahnya tepat = 1. Cross-entropy kemudian mengukur seberapa jauh distribusi probabilitas itu dari jawaban yang benar.

In [ ]:
def softmax_ce_loss_and_grad(z_out, y_int):
    """Hitung loss rata-rata dan gradient pertama (dL/dz_out).

    z_out : (B, C) - skor mentah dari output layer
    y_int : (B,)   - label benar sebagai integer (0 sampai C-1)
    """
    B, C = z_out.shape
    # Geser agar exp tidak overflow (stabilisasi numerik)
    z_shift = z_out - z_out.max(axis=1, keepdims=True)
    exp_z   = np.exp(z_shift)
    p       = exp_z / exp_z.sum(axis=1, keepdims=True)           # probabilitas softmax
    log_p   = z_shift - np.log(exp_z.sum(axis=1, keepdims=True))
    loss    = -log_p[np.arange(B), y_int].mean()                 # cross-entropy
    # Gradien gabungan softmax+CE: p - one_hot(y)
    grad = p.copy()
    grad[np.arange(B), y_int] -= 1.0
    grad /= B
    return loss, grad

In [ ]:
# Uji 3 situasi untuk memastikan fungsi bekerja masuk akal

# Situasi 1: model sangat yakin kelas 0 BENAR
z_yakin_benar = np.array([[10., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
L1, _ = softmax_ce_loss_and_grad(z_yakin_benar, np.array([0]))
print('Model yakin benar   --> loss:', round(L1, 4), '(harusnya sangat kecil)')

# Situasi 2: model sangat yakin kelas 1, padahal jawaban benar kelas 0
z_yakin_salah = np.array([[0., 10., 0., 0., 0., 0., 0., 0., 0., 0.]])
L2, _ = softmax_ce_loss_and_grad(z_yakin_salah, np.array([0]))
print('Model yakin salah   --> loss:', round(L2, 4), '(harusnya sangat besar)')

# Situasi 3: model tidak tahu sama sekali (semua skor = 0)
z_tidak_tahu = np.zeros((1, 10))
L3, _ = softmax_ce_loss_and_grad(z_tidak_tahu, np.array([0]))
print('Model tidak tahu    --> loss:', round(L3, 4), '(harusnya sekitar 2.30 = ln(10))')

### 2.3 Bobot dan Bias: Inisialisasi Parameter

Neural network punya dua jenis parameter: bobot (`W`) dan bias (`b`). Sebelum training, keduanya harus diisi dengan angka awal.

Kenapa tidak bisa diisi nol semua? Karena kalau semua bobot nol, semua neuron di layer yang sama menghitung hal yang sama dan mendapat gradient yang sama - mereka tidak pernah berbeda satu sama lain sepanjang training. Kita pakai Kaiming Normal agar magnitudo output tetap stabil di awal meski menggunakan ReLU.

In [ ]:
def init_params(d_in=784, d_h=256, d_out=10, seed=42):
    r = np.random.default_rng(seed)
    # Kaiming Normal: standar deviasi = sqrt(2 / fan_in), dirancang untuk ReLU
    W1 = r.standard_normal((d_h, d_in)).astype('float32') * np.sqrt(2.0 / d_in)
    b1 = np.zeros(d_h,   dtype='float32')
    W2 = r.standard_normal((d_out, d_h)).astype('float32') * np.sqrt(2.0 / d_h)
    b2 = np.zeros(d_out, dtype='float32')
    return {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}

p = init_params()
print('Parameter yang tersedia:')
for k, v in p.items():
    print('  ', k, '| ukuran:', v.shape, '| std:', round(float(v.std()), 4))

total = sum(v.size for v in p.values())
print()
print('Total parameter:', total, '= semua angka yang akan berubah saat training')

## Fase 3: Forward Pass dan Backward Pass

Komponen sudah selesai dibuat. Sekarang kita gabungkan menjadi dua fungsi utama: `forward` untuk menghitung output dari input, dan `backward` untuk menghitung gradient tiap parameter.

### 3.1 Forward Pass

Kita simpan semua hasil antara di dict `cache` karena backward pass membutuhkan nilai-nilai ini saat menghitung gradient.

In [ ]:
def forward(X, p):
    z1 = X @ p['W1'].T + p['b1']   # (B, 784) x (784, 256) = (B, 256)
    h  = relu(z1)                  # (B, 256), nilai negatif jadi 0
    z2 = h @ p['W2'].T + p['b2']   # (B, 256) x (256, 10)  = (B, 10)
    cache = {'X': X, 'z1': z1, 'h': h}
    return z2, cache

In [ ]:
z2_satu, cache_satu = forward(X_train[:1], p)

print('Input  shape:', X_train[:1].shape, '<-- 1 gambar, 784 piksel')
print('Output shape:', z2_satu.shape,     '<-- 10 skor mentah (satu per kelas)')
print()
print('Skor 10 kelas:', z2_satu.round(3))
print('Model sekarang menebak angka nomor:', z2_satu.argmax())
print('Jawaban yang benar                :', y_train[0])
print()
print('(Wajar jika masih salah - parameter belum dilatih sama sekali)')

### 3.2 Backward Pass

Backward pass menghitung gradient menggunakan chain rule, dari output ke input. Setiap parameter mendapat satu angka gradient: berapa perubahan loss jika parameter itu naik sedikit.

Ikuti 7 langkah dari Lampiran A.1. Tiap baris kode berkorespondensi dengan satu langkah dalam rantai turunan.

In [ ]:
def backward(grad_z2, cache, p):
    X, z1, h = cache['X'], cache['z1'], cache['h']

    # Langkah 2-3: cari gradient bobot dan bias layer 2
    gW2 = grad_z2.T @ h          # (10, B) x (B, 256) = (10, 256)
    gb2 = grad_z2.sum(axis=0)    # (10,)

    # Langkah 4: hitung gradient untuk layer tersembunyi
    gh = grad_z2 @ p['W2']       # (B, 10) x (10, 256) = (B, 256)

    # Langkah 5: terapkan turunan ReLU (di posisi yang nol saat forward, gradient juga nol)
    gz1 = gh * relu_grad(z1)     # (B, 256) elemen per elemen

    # Langkah 6-7: cari gradient bobot dan bias layer 1
    gW1 = gz1.T @ X              # (256, B) x (B, 784) = (256, 784)
    gb1 = gz1.sum(axis=0)        # (256,)

    return {'W1': gW1, 'b1': gb1, 'W2': gW2, 'b2': gb2}

## Fase 4: Verifikasi dan Loop Training

Sebelum melatih, kita verifikasi bahwa gradient yang dihitung `backward` sudah benar. Kemudian kita jalankan loop training langsung, tanpa membungkusnya dalam fungsi.

### 4.1 Verifikasi Gradient: Apakah Matematika Kita Benar?

Sebelum melatih, kita periksa apakah gradient dari `backward` sesuai dengan perkiraan numerik. Caranya: geser satu parameter sedikit (±ε), hitung perubahan loss, bandingkan dengan gradient analitik.

> [!NOTE]
> **Kenapa toleransi 1e-3 (bukan 1e-9)?** Finite-difference dengan `eps = 1e-4` menghitung `(L(W+eps) - L(W-eps)) / (2*eps)`. Kesalahan numerik dasar float32 (~1e-7) diperkuat oleh pembagian dengan `eps`, sehingga floor noise berada di sekitar `1e-7 / 1e-4 = 1e-3`. Selisih lebih kecil dari ini sudah masuk noise floating point; selisih lebih besar dari ini biasanya **bug** (salah turunan, salah axis sum, salah transpose). Aturan praktis: 1e-3 adalah threshold pragmatis untuk gradient check di float32; pakai `1e-5` kalau Anda kerja di float64.

In [ ]:
Xs = X_train[:8]
ys = y_train[:8]

z2, cache = forward(Xs, p)
loss, gz2 = softmax_ce_loss_and_grad(z2, ys)
grads = backward(gz2, cache, p)

eps      = 1e-4
max_diff = 0.0

print('=== Cek gradient W2 (5 posisi acak) ===')
W2 = p['W2']
for _ in range(5):
    i = rng.integers(W2.shape[0])
    j = rng.integers(W2.shape[1])
    orig      = W2[i, j]
    W2[i, j]  = orig + eps
    L_plus, _ = softmax_ce_loss_and_grad(forward(Xs, p)[0], ys)
    W2[i, j]  = orig - eps
    L_minus,_ = softmax_ce_loss_and_grad(forward(Xs, p)[0], ys)
    W2[i, j]  = orig
    g_num = (L_plus - L_minus) / (2 * eps)
    g_ana = grads['W2'][i, j]
    diff  = abs(g_num - g_ana)
    max_diff = max(max_diff, diff)
    print('W2[%d,%d]  numerik=%+.6f  analitik=%+.6f  |selisih|=%.2e' % (i, j, g_num, g_ana, diff))

print()
print('=== Cek gradient W1 (5 posisi acak) ===')
W1 = p['W1']
for _ in range(5):
    i = rng.integers(W1.shape[0])
    j = rng.integers(W1.shape[1])
    orig      = W1[i, j]
    W1[i, j]  = orig + eps
    L_plus, _ = softmax_ce_loss_and_grad(forward(Xs, p)[0], ys)
    W1[i, j]  = orig - eps
    L_minus,_ = softmax_ce_loss_and_grad(forward(Xs, p)[0], ys)
    W1[i, j]  = orig
    g_num = (L_plus - L_minus) / (2 * eps)
    g_ana = grads['W1'][i, j]
    diff  = abs(g_num - g_ana)
    max_diff = max(max_diff, diff)
    print('W1[%d,%d]  numerik=%+.6f  analitik=%+.6f  |selisih|=%.2e' % (i, j, g_num, g_ana, diff))

print()
print('Selisih terbesar: %.2e  (harus < 1e-3)' % max_diff)
if max_diff < 1e-3:
    print('Gradient check LULUS - matematika backward kita benar')
else:
    print('GAGAL - ada bug di backward pass')

### 4.2 Loop Training

Loop training ditulis langsung di sini, tidak dibungkus dalam fungsi `train()`. Setiap langkah terlihat secara eksplisit.

In [ ]:
# Inisialisasi ulang parameter yang bersih sebelum training
p          = init_params(seed=42)
lr         = 0.1
batch_size = 128
epochs     = 5
rng_train  = np.random.default_rng(42)

train_losses = []
test_accs    = []

for epoch in range(epochs):

    # Kocok urutan data agar batch tidak selalu sama tiap epoch
    idx        = rng_train.permutation(len(X_train))
    total_loss = 0.0

    for start in range(0, len(X_train), batch_size):

        # Langkah 1: ambil satu batch
        b       = idx[start : start + batch_size]
        Xb, yb  = X_train[b], y_train[b]

        # Langkah 2: forward pass - hitung output dari input
        z2, cache = forward(Xb, p)

        # Langkah 3: hitung loss dan gradient awal di output
        loss, grad_z2 = softmax_ce_loss_and_grad(z2, yb)

        # Langkah 4: backward pass - hitung gradient dari output ke input
        grads = backward(grad_z2, cache, p)

        # Langkah 5: update semua parameter (SGD)
        for k in p:
            p[k] -= lr * grads[k]

        total_loss += loss * len(b)

    avg_loss  = total_loss / len(X_train)
    pred_test = forward(X_test, p)[0].argmax(axis=1)
    acc       = (pred_test == y_test).mean()

    train_losses.append(avg_loss)
    test_accs.append(acc)
    print('Epoch %d/5  |  Loss: %.4f  |  Akurasi test: %.4f' % (epoch + 1, avg_loss, acc))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3))

ax[0].plot(train_losses, marker='o')
ax[0].set_title('Loss tiap epoch (NumPy MLP)')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Loss')

ax[1].plot(test_accs, marker='o', color='green')
ax[1].axhline(0.94, color='red', linestyle='--', alpha=0.5, label='Target >= 0.94')
ax[1].set_title('Akurasi di data uji')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Akurasi')
ax[1].legend()

plt.tight_layout()
plt.show()

print('Akurasi akhir:', round(test_accs[-1], 4))
if test_accs[-1] >= 0.94:
    print('Target tercapai! (>= 0.94)')
else:
    print('Di bawah target - periksa kembali backward pass dan gradient check di §4.1.')

**Apa yang Anda perhatikan?**

- **Train loss.** Apakah turun monoton, atau ada lonjakan? Lonjakan kecil di tengah epoch wajar (SGD bising); lonjakan tajam = lr terlalu besar.
- **Akurasi test.** Setelah 5 epoch dengan lr=0.1, akurasi target ≈ 0.94-0.96 di MNIST. Kalau di bawah 0.85, kemungkinan ada bug di `backward` (gradient check mungkin lulus pada subset kecil saja).
- **Bandingkan ke target produksi.** PyTorch `SimpleMLP` di §5 akan diuji di lr=0.1 yang sama. Kalau curve NumPy Anda **konsisten lebih buruk** (bukan sekadar offset kecil 0.01-0.02 yang masuk noise), kemungkinan bug di rumus turunan ReLU atau axis pada `gW1.T @ X`.

---

## Fase 5: Eksperimen Perubahan Komponen

Kita ubah satu komponen dan amati dampaknya pada gradient dan akurasi.

### Misi 1: Ganti ReLU dengan Sigmoid

Hipotesis: gradient W1 akan jauh lebih kecil dari W2. Ini terjadi karena turunan sigmoid bernilai paling besar 0.25, sehingga gradient mengecil setiap kali melewati satu layer ke belakang. Gejala ini disebut *vanishing gradient*.

Langkah yang harus dilakukan:
1. Jalankan sel di bawah untuk mendefinisikan `forward_sigmoid` dan `backward_sigmoid`.
2. Di sel berikutnya, jalankan satu mini-batch dan cetak magnitudo gradient W1 dan W2.
3. Tulis observasi di sel markdown yang disediakan.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_grad(z):
    s = sigmoid(z)
    return s * (1.0 - s)

def forward_sigmoid(X, p):
    z1    = X @ p['W1'].T + p['b1']
    h     = sigmoid(z1)              # <-- satu-satunya perubahan dari forward()
    z2    = h @ p['W2'].T + p['b2']
    cache = {'X': X, 'z1': z1, 'h': h}
    return z2, cache

def backward_sigmoid(grad_z2, cache, p):
    X, z1, h = cache['X'], cache['z1'], cache['h']
    gW2 = grad_z2.T @ h
    gb2 = grad_z2.sum(axis=0)
    gh  = grad_z2 @ p['W2']
    gz1 = gh * sigmoid_grad(z1)      # <-- satu-satunya perubahan dari backward()
    gW1 = gz1.T @ X
    gb1 = gz1.sum(axis=0)
    return {'W1': gW1, 'b1': gb1, 'W2': gW2, 'b2': gb2}

In [ ]:
# Bandingkan magnitudo gradient W1 vs W2 antara sigmoid dan ReLU
p_demo  = init_params(seed=42)
Xb_demo = X_train[:128]
yb_demo = y_train[:128]

# Dengan sigmoid
z2_sig, cache_sig = forward_sigmoid(Xb_demo, p_demo)
_, gz2_sig        = softmax_ce_loss_and_grad(z2_sig, yb_demo)
grads_sig         = backward_sigmoid(gz2_sig, cache_sig, p_demo)

# Dengan ReLU (menggunakan parameter yang sama)
z2_rel, cache_rel = forward(Xb_demo, p_demo)
_, gz2_rel        = softmax_ce_loss_and_grad(z2_rel, yb_demo)
grads_rel         = backward(gz2_rel, cache_rel, p_demo)

print('=== Magnitudo gradient (rata-rata nilai absolut) ===')
print()
gW1_sig = float(np.abs(grads_sig['W1']).mean())
gW2_sig = float(np.abs(grads_sig['W2']).mean())
gW1_rel = float(np.abs(grads_rel['W1']).mean())
gW2_rel = float(np.abs(grads_rel['W2']).mean())
print('Sigmoid  |  W1: %.6f  |  W2: %.6f' % (gW1_sig, gW2_sig))
print('ReLU     |  W1: %.6f  |  W2: %.6f' % (gW1_rel, gW2_rel))
print()
rasio = gW1_sig / (gW1_rel + 1e-10)
print('Rasio gradient W1 sigmoid/ReLU: %.3f' % rasio)
print('Semakin kecil dari 1.0, semakin parah vanishing gradient.')

**Observasi Misi 1:**

*(Tulis jawaban Anda di sini)*

- Apakah gradient W1 pada sigmoid jauh lebih kecil dari W2? Berapa rasionya?
- Apa konsekuensinya jika W1 mendapat gradient yang sangat kecil?

### Misi 2: Inisialisasi Nol - Ganti Kaiming dengan Nol

Hipotesis: kalau semua bobot diinisialisasi dengan nol, semua neuron di layer yang sama menghitung hal yang sama dan mendapat gradient yang sama - mereka tidak pernah berbeda satu sama lain. Akurasi akan stuck di sekitar 0.10 (tebak acak untuk 10 kelas).

In [ ]:
def init_params_nol(d_in=784, d_h=256, d_out=10):
    return {
        'W1': np.zeros((d_h,   d_in), dtype='float32'),
        'b1': np.zeros(d_h,          dtype='float32'),
        'W2': np.zeros((d_out, d_h),  dtype='float32'),
        'b2': np.zeros(d_out,         dtype='float32'),
    }

# Jalankan 2 epoch dengan inisialisasi nol
p_nol     = init_params_nol()
rng_nol   = np.random.default_rng(42)

for epoch in range(2):
    idx       = rng_nol.permutation(len(X_train))
    total_loss = 0.0
    for start in range(0, len(X_train), 128):
        b           = idx[start : start + 128]
        Xb, yb      = X_train[b], y_train[b]
        z2, cache   = forward(Xb, p_nol)
        loss, gz2   = softmax_ce_loss_and_grad(z2, yb)
        grads       = backward(gz2, cache, p_nol)
        for k in p_nol:
            p_nol[k] -= 0.1 * grads[k]
        total_loss  += loss * len(b)
    pred = forward(X_test, p_nol)[0].argmax(axis=1)
    acc  = (pred == y_test).mean()
    print('Epoch %d  |  Loss: %.4f  |  Akurasi: %.4f' % (epoch + 1, total_loss / len(X_train), acc))

print()
print('Pembanding: init Kaiming di §4.2 epoch 1 mencapai akurasi berapa?')

**Observasi Misi 2:**

*(Tulis jawaban Anda di sini)*

- Apakah akurasi benar-benar stuck di sekitar 0.10?
- Setelah 2 epoch, apakah W1 mulai berbeda antar neuron atau masih seragam?
- Mengapa inisialisasi nol menyebabkan semua neuron di satu layer mendapat gradient yang identik sepanjang training?

## Fase 6: Parity Check dengan PyTorch

Jalankan arsitektur yang identik melalui PyTorch autograd. Loss curve harus menyerupai curve manual di atas - bukan persis sama (seed RNG berbeda, batching berbeda), tapi urutan besaran dan titik konvergensi mirip.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# --- Auto-fetch src/ jika tidak tersedia (Colab-friendly) ---
import sys, os
_root = os.path.abspath('..')
if _root not in sys.path:
    sys.path.insert(0, _root)

SRC_URL = 'https://raw.githubusercontent.com/muhammad-zainal-muttaqin/Module-DS/master/template/src'

try:
    from src.models import SimpleMLP
except ModuleNotFoundError:
    print('src/ tidak ditemukan. Mengunduh dari GitHub...')
    import urllib.request
    urllib.request.urlretrieve(SRC_URL + '/models.py', 'models.py')
    from models import SimpleMLP
    print('Selesai.')

device     = 'cuda' if torch.cuda.is_available() else 'cpu'
torch_mdl  = SimpleMLP(input_dim=784, hidden_sizes=(256,), num_classes=10, activation='relu').to(device)
ds_tr      = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
dl_tr      = DataLoader(ds_tr, batch_size=128, shuffle=True)
opt        = optim.SGD(torch_mdl.parameters(), lr=0.1)
crit       = nn.CrossEntropyLoss()
torch_hist = []

for epoch in range(5):
    running = 0.0
    for Xb, yb in dl_tr:
        Xb, yb = Xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(torch_mdl(Xb), yb)
        loss.backward()
        opt.step()
        running += loss.item() * len(Xb)
    torch_hist.append(running / len(ds_tr))
    print('PyTorch epoch %d  |  loss=%.4f' % (epoch + 1, torch_hist[-1]))

plt.plot(train_losses, label='numpy manual', marker='o')
plt.plot(torch_hist,   label='pytorch autograd', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Train loss')
plt.legend()
plt.title('Manual NumPy vs PyTorch Autograd')
plt.show()

## Refleksi

### 7.1 Pertanyaan Tertarget

1. **Debugging journey.** Di langkah mana backward Anda pertama kali menemukan error? Pilih satu yang paling cocok dan tulis 2 kalimat detail:
   - (a) Shape mismatch saat menghitung `gW2.T @ h` atau sejenisnya.
   - (b) Loss menjadi NaN setelah beberapa iterasi.
   - (c) Gradient stuck nol di `gz1` (kemungkinan semua nilai `z1` negatif, sehingga `relu_grad` nol di seluruh batch).
   - (d) Gradient check `|diff| > 1e-3` walaupun shape benar.
   - (e) Tidak ada masalah - berhasil dari percobaan pertama.

2. **Eksperimen sigmoid.** Dari Misi 1 di §5, apakah curve loss sigmoid turun lebih lambat dari ReLU? Sebutkan satu kondisi di mana sigmoid masih pilihan yang masuk akal meski vanishing gradient lebih parah.

3. **Threshold gradient check.** Kenapa finite-difference tidak pernah persis sama dengan analitik? Sebutkan dua sumber error numerik selain yang sudah disebut di catatan §4.1. Kapan gap mulai jadi indikator bug, bukan sekadar noise?

### 7.2 Self-Check Quick

- [ ] Forward pass shape match: input `(B, 784)` menghasilkan output `(B, 10)`.
- [ ] Gradient check `max diff < 1e-3` di 5 posisi acak.
- [ ] Training 5 epoch selesai tanpa NaN; akurasi test >= 0.94.
- [ ] Plot parity NumPy vs PyTorch tumpang tindih dalam toleransi 0.05.
- [ ] Saya bisa menjelaskan kenapa `softmax + CE` digabung menghasilkan gradient `p - y_onehot`.